# Unified ASR Model Comparison for Twi
## Comprehensive Benchmarking of Faster-Whisper, MMS, Wav2Vec2, and Moshi

This notebook provides a unified framework to train, test, and compare all ASR models on Twi data.

In [ ]:
# Installation
!pip install -q transformers datasets torch torchaudio evaluate jiwer
!pip install -q faster-whisper librosa soundfile pandas matplotlib seaborn
!pip install -q tabulate scikit-learn

import os
import sys
import json
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, asdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Audio
import librosa
import soundfile as sf

# Metrics
import evaluate
from sklearn.metrics import accuracy_score

# Check environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")

print(f"\n🌍 Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

# Auto-detect and use correct data path
if IN_COLAB:
    # Mount Google Drive if in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive/MyDrive/voice-lab-data')
elif IN_KAGGLE:
    DATA_PATH = Path('/kaggle/working/voice-lab-data')
else:
    # Local environment
    DATA_PATH = Path('/opt/kasaai/apps/voice-lab/data')

print(f"📂 Data path: {DATA_PATH}")

# Check if processed data exists
metadata_path = DATA_PATH / 'metadata/all_datasets_processed.csv'
if metadata_path.exists():
    processed_data = pd.read_csv(metadata_path)
    print(f"✅ Found processed data: {len(processed_data)} samples")
    print(f"   Languages: {processed_data['language'].unique() if 'language' in processed_data.columns else 'N/A'}")
    print(f"   Datasets: {processed_data['dataset'].unique() if 'dataset' in processed_data.columns else 'N/A'}")
else:
    print(f"⚠️ Processed data not found at {metadata_path}")
    print("   Please run notebooks 001 and 002 first to download and process data")

In [ ]:
# Unified configuration
@dataclass
class UnifiedConfig:
    """Configuration for all models."""
    # Data paths - use global DATA_PATH
    data_dir: Path = DATA_PATH
    models_dir: Path = DATA_PATH.parent / "models"
    results_dir: Path = DATA_PATH.parent / "results"
    
    # Audio settings
    sample_rate: int = 16000
    max_duration: float = 30.0
    
    # Evaluation settings
    test_size: int = 100  # Number of test samples
    beam_size: int = 5
    
    # Model configurations
    models: Dict = None
    
    def __post_init__(self):
        self.results_dir.mkdir(parents=True, exist_ok=True)
        
        # Define model configurations
        self.models = {
            'faster_whisper': {
                'name': 'Faster-Whisper',
                'model_path': 'openai/whisper-small',
                'type': 'whisper',
                'color': '#FF6B6B'
            },
            'mms': {
                'name': 'Meta MMS',
                'model_path': 'facebook/mms-1b-all',
                'type': 'mms',
                'color': '#4ECDC4'
            },
            'wav2vec2': {
                'name': 'Wav2Vec2',
                'model_path': 'facebook/wav2vec2-large-xlsr-53',
                'type': 'wav2vec2',
                'color': '#45B7D1'
            },
            'wav2vec2_akan': {
                'name': 'Wav2Vec2-Akan',
                'model_path': 'azunre/wav2vec2-akan',
                'type': 'wav2vec2',
                'color': '#96CEB4'
            },
            'moshi': {
                'name': 'Kyutai Moshi',
                'model_path': 'kyutai/moshi',  # Hypothetical
                'type': 'moshi',
                'color': '#DDA0DD'
            }
        }

config = UnifiedConfig()
print(f"📋 Unified configuration loaded for {len(config.models)} models")
print(f"   Data directory: {config.data_dir}")
print(f"   Models directory: {config.models_dir}")
print(f"   Results directory: {config.results_dir}")

In [ ]:
# Model loader interface
class ASRModelInterface:
    """Unified interface for all ASR models."""
    
    def __init__(self, model_config: Dict, device: str = 'cuda'):
        self.config = model_config
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.model = None
        self.processor = None
        
    def load_model(self):
        """Load model based on type."""
        raise NotImplementedError
        
    def transcribe(self, audio_path: str) -> str:
        """Transcribe audio file."""
        raise NotImplementedError
        
    def batch_transcribe(self, audio_paths: List[str]) -> List[str]:
        """Transcribe multiple audio files."""
        return [self.transcribe(path) for path in audio_paths]

In [ ]:
# Whisper model wrapper
class WhisperModel(ASRModelInterface):
    """Wrapper for Whisper/Faster-Whisper models."""
    
    def load_model(self):
        try:
            # Try Faster-Whisper first
            from faster_whisper import WhisperModel as FW_Model
            self.model = FW_Model(
                self.config['model_path'],
                device=self.device,
                compute_type="int8_float16" if self.device == "cuda" else "int8"
            )
            self.use_faster = True
            print(f"✅ Loaded Faster-Whisper: {self.config['name']}")
        except:
            # Fallback to Transformers
            from transformers import WhisperForConditionalGeneration, WhisperProcessor
            self.processor = WhisperProcessor.from_pretrained(self.config['model_path'])
            self.model = WhisperForConditionalGeneration.from_pretrained(self.config['model_path'])
            if self.device == 'cuda':
                self.model = self.model.cuda()
            self.use_faster = False
            print(f"✅ Loaded Transformers Whisper: {self.config['name']}")
    
    def transcribe(self, audio_path: str) -> str:
        audio, sr = librosa.load(audio_path, sr=16000)
        
        if self.use_faster:
            segments, _ = self.model.transcribe(
                audio,
                beam_size=5,
                language="tw"
            )
            return " ".join([s.text for s in segments])
        else:
            inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
            if self.device == 'cuda':
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            with torch.no_grad():
                predicted_ids = self.model.generate(**inputs)
            
            return self.processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

In [ ]:
# MMS model wrapper
class MMSModel(ASRModelInterface):
    """Wrapper for Meta MMS model."""
    
    def load_model(self):
        from transformers import Wav2Vec2ForCTC, AutoProcessor
        
        self.processor = AutoProcessor.from_pretrained(self.config['model_path'])
        self.model = Wav2Vec2ForCTC.from_pretrained(
            self.config['model_path'],
            ignore_mismatched_sizes=True
        )
        
        # Set language
        self.processor.tokenizer.set_target_lang("twi")
        self.model.load_adapter("twi")
        
        if self.device == 'cuda':
            self.model = self.model.cuda()
        
        print(f"✅ Loaded MMS: {self.config['name']}")
    
    def transcribe(self, audio_path: str) -> str:
        audio, sr = librosa.load(audio_path, sr=16000)
        
        inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
        if self.device == 'cuda':
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
        
        predicted_ids = torch.argmax(logits, dim=-1)
        return self.processor.batch_decode(predicted_ids)[0]

In [ ]:
# Wav2Vec2 model wrapper
class Wav2Vec2Model(ASRModelInterface):
    """Wrapper for Wav2Vec2 models."""
    
    def load_model(self):
        from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
        
        try:
            self.processor = Wav2Vec2Processor.from_pretrained(self.config['model_path'])
            self.model = Wav2Vec2ForCTC.from_pretrained(self.config['model_path'])
            
            if self.device == 'cuda':
                self.model = self.model.cuda()
            
            print(f"✅ Loaded Wav2Vec2: {self.config['name']}")
        except Exception as e:
            print(f"❌ Could not load {self.config['name']}: {e}")
            self.model = None
    
    def transcribe(self, audio_path: str) -> str:
        if self.model is None:
            return "[Model not available]"
        
        audio, sr = librosa.load(audio_path, sr=16000)
        
        inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
        if self.device == 'cuda':
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
        
        predicted_ids = torch.argmax(logits, dim=-1)
        return self.processor.batch_decode(predicted_ids)[0]

In [ ]:
# Model factory
class ModelFactory:
    """Factory to create appropriate model wrappers."""
    
    @staticmethod
    def create_model(model_key: str, config: UnifiedConfig) -> ASRModelInterface:
        model_config = config.models[model_key]
        model_type = model_config['type']
        
        if model_type == 'whisper':
            return WhisperModel(model_config)
        elif model_type == 'mms':
            return MMSModel(model_config)
        elif model_type == 'wav2vec2':
            return Wav2Vec2Model(model_config)
        elif model_type == 'moshi':
            # Return dummy model for now
            print(f"⚠️ Moshi model not yet implemented, using dummy")
            return ASRModelInterface(model_config)
        else:
            raise ValueError(f"Unknown model type: {model_type}")

# Load all models
models = {}
for model_key in config.models.keys():
    try:
        model = ModelFactory.create_model(model_key, config)
        model.load_model()
        models[model_key] = model
    except Exception as e:
        print(f"⚠️ Could not load {model_key}: {e}")

print(f"\n✅ Loaded {len(models)} models successfully")

In [ ]:
# Test data loader
class TestDataLoader:
    """Load test data for evaluation."""
    
    def __init__(self, config: UnifiedConfig):
        self.config = config
        
    def load_test_data(self) -> pd.DataFrame:
        """Load test dataset."""
        test_data = []
        
        # Try to load from metadata
        metadata_path = self.config.data_dir / "metadata/all_datasets_processed.csv"
        if metadata_path.exists():
            df = pd.read_csv(metadata_path)
            
            if 'audio_path' in df.columns and 'processed_text' in df.columns:
                df = df.dropna(subset=['audio_path', 'processed_text'])
                
                # Update audio paths to be absolute
                df['audio'] = df['audio_path'].apply(
                    lambda x: str(self.config.data_dir / 'audio' / Path(x).name) if pd.notna(x) else None
                )
                
                # Check file existence
                df['exists'] = df['audio'].apply(lambda x: Path(x).exists() if pd.notna(x) else False)
                existing_df = df[df['exists']]
                
                if len(existing_df) > 0:
                    # Sample test data
                    test_df = existing_df.sample(min(self.config.test_size, len(existing_df)), random_state=42)
                    test_data = test_df[['audio', 'processed_text']].rename(
                        columns={'processed_text': 'reference'}
                    )
                    print(f"✅ Loaded {len(test_data)} test samples from processed data")
                    return test_data
                else:
                    print(f"⚠️ No audio files found at expected paths")
        
        # Create dummy test data
        print("⚠️ No real test data available, creating dummy data for testing")
        
        # Create dummy audio files
        dummy_dir = self.config.data_dir / "dummy_audio"
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        dummy_samples = [
            ("Wo ho te sɛn", "test1.wav"),
            ("Me ma wo akye", "test2.wav"),
            ("Me da wo ase", "test3.wav"),
            ("Ɛyɛ sɛn", "test4.wav"),
            ("Nante yie", "test5.wav")
        ]
        
        dummy_data = []
        for text, filename in dummy_samples:
            audio_file = dummy_dir / filename
            if not audio_file.exists():
                # Generate 1 second of silence
                dummy_audio = np.zeros(16000, dtype=np.float32)
                sf.write(audio_file, dummy_audio, 16000)
            
            dummy_data.append({
                "audio": str(audio_file),
                "reference": text
            })
        
        return pd.DataFrame(dummy_data)

# Load test data
data_loader = TestDataLoader(config)
test_data = data_loader.load_test_data()
print(f"\n📊 Test dataset: {len(test_data)} samples")
print(f"   First few samples:")
for idx, row in test_data.head(3).iterrows():
    print(f"   - {Path(row['audio']).name}: {row['reference']}")

In [ ]:
# Evaluation metrics
class MetricsCalculator:
    """Calculate various ASR metrics."""
    
    def __init__(self):
        self.wer_metric = evaluate.load("wer")
        self.cer_metric = evaluate.load("cer")
        
    def calculate_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Calculate all metrics."""
        # Clean predictions and references
        predictions = [p.strip() for p in predictions]
        references = [r.strip() for r in references]
        
        # Calculate metrics
        wer = self.wer_metric.compute(predictions=predictions, references=references)
        cer = self.cer_metric.compute(predictions=predictions, references=references)
        
        # Calculate additional metrics
        exact_match = sum(p == r for p, r in zip(predictions, references)) / len(predictions)
        
        # Average length difference
        length_diffs = [abs(len(p.split()) - len(r.split())) for p, r in zip(predictions, references)]
        avg_length_diff = np.mean(length_diffs)
        
        return {
            'wer': wer,
            'cer': cer,
            'exact_match': exact_match,
            'avg_length_diff': avg_length_diff
        }

metrics_calc = MetricsCalculator()
print("✅ Metrics calculator initialized")

In [ ]:
# Benchmark runner
class BenchmarkRunner:
    """Run benchmarks for all models."""
    
    def __init__(self, models: Dict, test_data: pd.DataFrame, metrics_calc: MetricsCalculator):
        self.models = models
        self.test_data = test_data
        self.metrics_calc = metrics_calc
        self.results = {}
        
    def run_benchmark(self, model_key: str) -> Dict:
        """Run benchmark for a single model."""
        print(f"\n🏃 Benchmarking {config.models[model_key]['name']}...")
        
        model = self.models[model_key]
        predictions = []
        references = []
        inference_times = []
        
        for _, row in tqdm(self.test_data.iterrows(), total=len(self.test_data)):
            audio_path = row['audio']
            reference = row['reference']
            
            # Skip if audio doesn't exist
            if not Path(audio_path).exists():
                predictions.append("")
                references.append(reference)
                inference_times.append(0)
                continue
            
            # Measure inference time
            start_time = time.time()
            
            try:
                prediction = model.transcribe(audio_path)
            except Exception as e:
                print(f"  ⚠️ Error: {e}")
                prediction = ""
            
            inference_time = time.time() - start_time
            
            predictions.append(prediction)
            references.append(reference)
            inference_times.append(inference_time)
        
        # Calculate metrics
        metrics = self.metrics_calc.calculate_metrics(predictions, references)
        
        # Add timing metrics
        metrics['avg_inference_time'] = np.mean(inference_times)
        metrics['total_time'] = sum(inference_times)
        metrics['rtf'] = metrics['avg_inference_time'] / 1.0  # Assuming 1 second audio
        
        # Store results
        self.results[model_key] = {
            'metrics': metrics,
            'predictions': predictions[:5],  # Store sample predictions
            'references': references[:5]
        }
        
        print(f"  WER: {metrics['wer']:.2%}")
        print(f"  CER: {metrics['cer']:.2%}")
        print(f"  Avg time: {metrics['avg_inference_time']:.3f}s")
        
        return metrics
    
    def run_all_benchmarks(self):
        """Run benchmarks for all models."""
        print("\n🚀 Starting comprehensive benchmark...")
        print(f"   Models: {len(self.models)}")
        print(f"   Test samples: {len(self.test_data)}")
        
        for model_key in self.models.keys():
            self.run_benchmark(model_key)
        
        print("\n✅ All benchmarks complete!")
        return self.results

# Run benchmarks
benchmark = BenchmarkRunner(models, test_data, metrics_calc)
results = benchmark.run_all_benchmarks()

In [ ]:
# Results visualization
class ResultsVisualizer:
    """Visualize benchmark results."""
    
    def __init__(self, results: Dict, config: UnifiedConfig):
        self.results = results
        self.config = config
        
    def create_comparison_table(self) -> pd.DataFrame:
        """Create comparison table."""
        data = []
        
        for model_key, result in self.results.items():
            metrics = result['metrics']
            data.append({
                'Model': self.config.models[model_key]['name'],
                'WER (%)': f"{metrics['wer']*100:.1f}",
                'CER (%)': f"{metrics['cer']*100:.1f}",
                'Exact Match (%)': f"{metrics['exact_match']*100:.1f}",
                'Avg Time (s)': f"{metrics['avg_inference_time']:.3f}",
                'RTF': f"{metrics['rtf']:.2f}"
            })
        
        df = pd.DataFrame(data)
        return df
    
    def plot_metrics_comparison(self):
        """Plot metrics comparison."""
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # Extract data
        model_names = [self.config.models[k]['name'] for k in self.results.keys()]
        colors = [self.config.models[k]['color'] for k in self.results.keys()]
        
        # WER comparison
        wer_scores = [r['metrics']['wer']*100 for r in self.results.values()]
        axes[0, 0].bar(model_names, wer_scores, color=colors)
        axes[0, 0].set_title('Word Error Rate (WER)', fontsize=14, fontweight='bold')
        axes[0, 0].set_ylabel('WER (%)')
        axes[0, 0].set_ylim(0, max(wer_scores) * 1.2 if wer_scores else 100)
        
        # CER comparison
        cer_scores = [r['metrics']['cer']*100 for r in self.results.values()]
        axes[0, 1].bar(model_names, cer_scores, color=colors)
        axes[0, 1].set_title('Character Error Rate (CER)', fontsize=14, fontweight='bold')
        axes[0, 1].set_ylabel('CER (%)')
        axes[0, 1].set_ylim(0, max(cer_scores) * 1.2 if cer_scores else 100)
        
        # Inference time comparison
        times = [r['metrics']['avg_inference_time'] for r in self.results.values()]
        axes[1, 0].bar(model_names, times, color=colors)
        axes[1, 0].set_title('Average Inference Time', fontsize=14, fontweight='bold')
        axes[1, 0].set_ylabel('Time (seconds)')
        
        # Exact match comparison
        exact_match = [r['metrics']['exact_match']*100 for r in self.results.values()]
        axes[1, 1].bar(model_names, exact_match, color=colors)
        axes[1, 1].set_title('Exact Match Rate', fontsize=14, fontweight='bold')
        axes[1, 1].set_ylabel('Exact Match (%)')
        
        # Rotate x-axis labels
        for ax in axes.flat:
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3)
        
        plt.suptitle('ASR Model Comparison for Twi', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        # Save figure
        plt.savefig(self.config.results_dir / 'model_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def plot_speed_accuracy_tradeoff(self):
        """Plot speed vs accuracy tradeoff."""
        plt.figure(figsize=(10, 6))
        
        for model_key, result in self.results.items():
            wer = result['metrics']['wer'] * 100
            speed = 1 / result['metrics']['avg_inference_time']  # Samples per second
            color = self.config.models[model_key]['color']
            name = self.config.models[model_key]['name']
            
            plt.scatter(speed, 100 - wer, s=200, c=color, label=name, alpha=0.7)
            plt.annotate(name, (speed, 100 - wer), fontsize=9, ha='center')
        
        plt.xlabel('Speed (samples/second)', fontsize=12)
        plt.ylabel('Accuracy (100 - WER%)', fontsize=12)
        plt.title('Speed vs Accuracy Tradeoff', fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.savefig(self.config.results_dir / 'speed_accuracy_tradeoff.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def show_sample_predictions(self):
        """Show sample predictions from each model."""
        print("\n📝 Sample Predictions:")
        print("=" * 80)
        
        # Get first reference
        first_result = list(self.results.values())[0]
        num_samples = min(3, len(first_result['references']))
        
        for i in range(num_samples):
            print(f"\nSample {i+1}:")
            print(f"Reference: {first_result['references'][i]}")
            print("-" * 40)
            
            for model_key, result in self.results.items():
                model_name = self.config.models[model_key]['name']
                prediction = result['predictions'][i] if i < len(result['predictions']) else "N/A"
                print(f"{model_name:15}: {prediction}")

# Visualize results
visualizer = ResultsVisualizer(results, config)

# Create comparison table
comparison_df = visualizer.create_comparison_table()
print("\n📊 Model Comparison Table:")
print(tabulate(comparison_df, headers='keys', tablefmt='grid', showindex=False))

# Plot comparisons
visualizer.plot_metrics_comparison()
visualizer.plot_speed_accuracy_tradeoff()
visualizer.show_sample_predictions()

In [ ]:
# Save results
class ResultsSaver:
    """Save benchmark results."""
    
    def __init__(self, results: Dict, config: UnifiedConfig):
        self.results = results
        self.config = config
        
    def save_json(self):
        """Save results as JSON."""
        json_path = self.config.results_dir / 'benchmark_results.json'
        
        # Convert results to serializable format
        serializable_results = {}
        for model_key, result in self.results.items():
            serializable_results[model_key] = {
                'model_name': self.config.models[model_key]['name'],
                'metrics': result['metrics'],
                'sample_predictions': list(zip(
                    result['references'][:5],
                    result['predictions'][:5]
                ))
            }
        
        with open(json_path, 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        print(f"💾 Results saved to {json_path}")
    
    def save_csv(self):
        """Save metrics as CSV."""
        csv_path = self.config.results_dir / 'benchmark_metrics.csv'
        
        data = []
        for model_key, result in self.results.items():
            row = {'model': self.config.models[model_key]['name']}
            row.update(result['metrics'])
            data.append(row)
        
        df = pd.DataFrame(data)
        df.to_csv(csv_path, index=False)
        
        print(f"💾 Metrics saved to {csv_path}")
    
    def generate_report(self):
        """Generate markdown report."""
        report_path = self.config.results_dir / 'benchmark_report.md'
        
        with open(report_path, 'w') as f:
            f.write("# ASR Model Benchmark Report for Twi\n\n")
            f.write(f"**Date**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n\n")
            
            # Summary
            f.write("## Summary\n\n")
            f.write(f"- **Models tested**: {len(self.results)}\n")
            f.write(f"- **Test samples**: {config.test_size}\n")
            f.write(f"- **Best WER**: {min(r['metrics']['wer'] for r in self.results.values()):.2%}\n")
            f.write(f"- **Fastest model**: {min(self.results.items(), key=lambda x: x[1]['metrics']['avg_inference_time'])[0]}\n\n")
            
            # Detailed results
            f.write("## Detailed Results\n\n")
            
            for model_key, result in self.results.items():
                model_name = self.config.models[model_key]['name']
                metrics = result['metrics']
                
                f.write(f"### {model_name}\n\n")
                f.write(f"- **WER**: {metrics['wer']:.2%}\n")
                f.write(f"- **CER**: {metrics['cer']:.2%}\n")
                f.write(f"- **Exact Match**: {metrics['exact_match']:.2%}\n")
                f.write(f"- **Avg Inference Time**: {metrics['avg_inference_time']:.3f}s\n")
                f.write(f"- **RTF**: {metrics['rtf']:.2f}\n\n")
            
            # Recommendations
            f.write("## Recommendations\n\n")
            
            # Find best accuracy
            best_accuracy = min(self.results.items(), key=lambda x: x[1]['metrics']['wer'])
            f.write(f"- **Best accuracy**: {self.config.models[best_accuracy[0]]['name']} (WER: {best_accuracy[1]['metrics']['wer']:.2%})\n")
            
            # Find best speed
            best_speed = min(self.results.items(), key=lambda x: x[1]['metrics']['avg_inference_time'])
            f.write(f"- **Best speed**: {self.config.models[best_speed[0]]['name']} ({best_speed[1]['metrics']['avg_inference_time']:.3f}s/sample)\n")
            
            # Best balance
            scores = {}
            for k, v in self.results.items():
                # Balance score: lower is better
                scores[k] = v['metrics']['wer'] + (v['metrics']['avg_inference_time'] / 10)
            best_balance = min(scores.items(), key=lambda x: x[1])
            f.write(f"- **Best balance**: {self.config.models[best_balance[0]]['name']}\n")
        
        print(f"📄 Report saved to {report_path}")

# Save all results
saver = ResultsSaver(results, config)
saver.save_json()
saver.save_csv()
saver.generate_report()

## Summary

This unified comparison notebook provides:

### Features:
1. **Unified Interface**: Single interface for all ASR models
2. **Comprehensive Metrics**: WER, CER, speed, exact match
3. **Visualization**: Charts comparing all models
4. **Speed-Accuracy Tradeoff**: Analysis of efficiency vs performance
5. **Automated Reports**: JSON, CSV, and Markdown outputs

### Models Compared:
- **Faster-Whisper**: OpenAI's optimized Whisper
- **Meta MMS**: Massively multilingual model
- **Wav2Vec2**: Self-supervised models
- **Wav2Vec2-Akan**: Akan-specific model
- **Kyutai Moshi**: Next-gen architecture

### Key Insights:
- Compare accuracy (WER/CER) across models
- Evaluate inference speed and efficiency
- Identify best model for specific use cases
- Understand tradeoffs between speed and accuracy

### Next Steps:
1. Train models on full Twi dataset
2. Add more evaluation metrics
3. Test on diverse audio conditions
4. Optimize best performers for production